# HeatUp 02 - compute (optional)

Loads the interatomic potential and runs **everything the plans ask for** (`plan.json` of every phase, written by
notebook 01) that is not done yet: every step of every phase, in dependency order (relax, elastic, phonons, qha, fc3, md). The
relaxation has normally been done in notebook 01 already (it shows as `done`). Nothing about the
calculations is configured here - only the calculator. Skip this notebook if you compute the results yourself (e.g.
with DFT in the same folders): notebook 03 only needs the files.

Each task runs in its own process (GPU memory is released); a task is run when it is *pending*, *stale*, *unfinished*
(an interrupted MD resumes) or *failed* (retried); finished work is never repeated and a failure never stops the others.
Safe to run again at any time.

In [ ]:
import os
import warnings
import pandas as pd
from IPython.display import display

import heatup
from heatup import cfg, run_plan, status
from heatup.analyze.report import show_status, status_table

warnings.filterwarnings("ignore")
heatup.verbose()
if os.path.exists("heatup_project.json"):
    heatup.load_json("heatup_project.json")  # folder roots written by notebook 01
print("HeatUp", heatup.__version__, "|", cfg.phases.database_root)

## Configuration - the calculator
Loaded from `heatup_project.json` (set in notebook 01: the relaxation there and the steps here must use the same one). Override only on purpose: a different calculator makes finished steps *stale* and recomputes them.

In [ ]:
print("calculator:", cfg.calc.backend, cfg.calc.model, "| dtype", cfg.calc.dtype)
# cfg.calc.backend = "chgnet"        # ... overrides, see notebook 01 for the options
DEVICE = None                        # None -> cuda if usable, else cpu ('cpu' / 'cuda' to force)

cfg.run.force = []  # recompute whatever the cache says: ["md"], ["relax", "elastic"] or ["all"]

## What is left to do
The same table notebook 03 uses to warn about unfinished work (`calc=True` also compares the calculator).

In [ ]:
rows = status(calc=True)
show_status(rows)
todo = run_plan(dry_run=True)
display(pd.DataFrame(todo).query("action != 'done'").set_index("phase"))

## Run
Everything that is planned:

In [ ]:
results = run_plan(device=DEVICE)
display(pd.DataFrame(results).groupby("action").size().rename("tasks").to_frame())

### Your own protocol (optional)
`run_plan` is one simple loop, so any protocol is just several calls, e.g. a cheap first stage you inspect before paying for the rest:
```python
run_plan(steps=["relax", "elastic"])          # only these steps (of every phase)
status()                                      # look at the results, edit plan.json / drop phases ...
run_plan()                                    # ... then everything else
run_plan(targets=["database/BaTiO3/P4mm"])    # only some targets (and their competitors)
```
(`cfg.plan.prune_window_eV` in notebook 01 is the one built-in shortcut of this kind, and it is off by default.)

## Afterwards

In [ ]:
show_status(status(calc=True))
display(status_table(status(calc=True), problems_only=True))

### Forcing a rerun / provenance
`cfg.run.force` (above) or `with override(**{"run.force": ["relax"]}): run_plan()` recomputes steps whatever the cache says.
Every finished step stores the exact options it used:

In [ ]:
from heatup.core.ledger import manifest_summary
tag = results[0]["phase"]
print(manifest_summary(os.path.join(cfg.phases.database_root, tag), "relax"))